In [55]:
import glob
import os

import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO
import pandas as pd
from matplotlib.patches import Rectangle, Polygon

import sys
sys.path.append("../../../researches/objectdetectionmetrics/")
from metricsevaluator import MetricsEvaluator

sys.path.append("../../../PG_PyTraining")
from common.pg.pg_geometry import PgCamera

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
# font 
font = cv2.FONT_HERSHEY_SIMPLEX 
  
# fontScale 
fontScale = 0.35
  
# Line thickness of 2 px 
thickness = 1

In [57]:
def process_prediction(predictions):
    labels = predictions.boxes.cls.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2]
    confidences = predictions.boxes.conf.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2]
    boxes = predictions.boxes.xywhn.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2]
    points = predictions.keypoints.xyn.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2].reshape((-1, 2))
    
    return os.path.basename(predictions.path).split(".")[0], labels, confidences, boxes, points

In [58]:
def parse_img_name(img_name):
    h = 0
    sensor = ""
    success = False
    for i in img_name.split("_"):
        if i.startswith("H"):
            h = int(i[1:])
        elif i.startswith("Cam"):
            sensor = i
        
        if h > 0 and len(sensor) > 1:
            success = True
            break
    
    return success, h, sensor

In [59]:
base_path = "/home/klymy/data/ObjectDetectionV3_2"
df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture_table_top.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped_th_0.65_holdout.csv"))
df_imgs = df_imgs[df_imgs["group_id"] == "holdout"]
df = df[df["occupied_by_person"] == False]
df_imgs.head()

/tmp/ipykernel_530902/1601824404.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture_table_top.csv"))


,id,name,width,height,batch,folder,task_id,job_id,job_url,group_id
260,520b7950-dee4-454b-8afc-13fb6e8f19bc,Cam3_H283_menivimks_objects_1_2024-03-04-10h24...,640,480,annotation einav,einav6,658844,837576,https://app.cvat.ai/tasks/658844/jobs/837576,holdout
261,9cf116dc-c4b0-4ad1-b037-e34da8bb81dc,Cam3_H283_menivimks_objects_1_2024-03-04-10h24...,640,480,annotation einav,einav6,658844,837576,https://app.cvat.ai/tasks/658844/jobs/837576,holdout
271,65126fe3-410f-41df-945e-49c7e78579d2,H240_cam2_OpenSpace_10fps_1.avi_snapshot_00.01...,640,480,annotation einav,einav6,658844,837576,https://app.cvat.ai/tasks/658844/jobs/837576,holdout
272,c798c498-499d-4830-8d67-6a99a94a3567,H240_cam2_OpenSpace_10fps_1.avi_snapshot_00.01...,640,480,annotation einav,einav6,658844,837576,https://app.cvat.ai/tasks/658844/jobs/837576,holdout
273,efe5f873-da60-414c-9f16-bfe5df8b6c1c,H240_cam2_OpenSpace_10fps_2.avi_snapshot_00.01...,640,480,annotation einav,einav6,658844,837576,https://app.cvat.ai/tasks/658844/jobs/837576,holdout


In [60]:
df.head()

,label,not_ground,truncated,occluded,occupied_by_person,occupied,ID,xtl,ytl,xbr,ybr,x,y,img_id,id,corners
0,monitor,True,True,False,False,False,5,145.61,420.30,325.36,478.52,233.21,457.60,bdd640fb-0667-4ad1-9c80-317fa3b1799d,23b8c1e9-3924-46de-beb1-3b9046685257,NaN
1,chair,False,False,True,False,False,4,306.57,316.82,390.56,422.84,343.68,365.22,bdd640fb-0667-4ad1-9c80-317fa3b1799d,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,NaN
2,chair,False,False,True,False,False,3,198.15,225.50,284.59,299.78,238.68,275.36,bdd640fb-0667-4ad1-9c80-317fa3b1799d,972a8469-1641-4f82-8b9d-2434e465e150,NaN
3,chair,False,False,True,False,False,2,127.83,309.99,215.73,384.26,177.15,342.27,bdd640fb-0667-4ad1-9c80-317fa3b1799d,9a1de644-815e-46d1-bb8f-aa1837f8a88b,NaN
4,table,False,False,False,False,False,1,214.64,298.35,305.78,425.78,261.06,364.18,bdd640fb-0667-4ad1-9c80-317fa3b1799d,17fc695a-07a0-4a6e-8822-e8f36c031199,"[299.8, 306.5, 218.0, 306.1, 303.88, 421.14, 2..."


In [61]:


INPUT_WIDTH = 640
INPUT_HEIGHT = 640

classes = ["chair", "table"]

gt_color = (0, 255, 0)

# colors = {
#     "chair": (255, 0, 0),
#     "table": (0, 0, 255)
# }
colors = [(128, 0, 0), (0, 0, 128)]


base_pth = "../results"
model_name = "v3_2_table_chair_table_top_v2_no_occupied_chair"

thres = 0.35
iou_thres = 0.45

In [62]:
def draw_gt(img, gt_data):
    for _, row in gt_data.iterrows():
        cv2.rectangle(
            img,
            (int(row["xtl"]), int(row["ytl"])),
            (int(row["xbr"]), int(row["ybr"])),
            gt_color,
            3
        )

In [63]:
def draw_pred(img, labels, confidences, boxes):
    img_h, img_w, _ = img.shape
    for l, c, b in zip(labels, confidences, boxes):
        x, y, w, h = b
        x1 = int((x - w / 2) * img_w)
        y1 = int((y - h / 2) * img_h)
        x2 = x1 + int(w * img_w)
        y2 = y1 + int(h * img_h)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            colors[int(l)],
            2
        )

In [90]:
def draw_floorplan(ax, pg_camera, img, points, bboxes, labels):
    # # fig, ax = plt.subplots() 
    # plt.xlim(-500, 500)
    # plt.ylim(-200, 200)
    # plt.axis("equal")

    img_h, img_w, _ = img.shape
    points = points[labels == 0]
    points *= [img_w, img_h]
    points_cm = pg_camera.pixel2cm(points)

    plt.plot(0, 0, "k.", markersize=15)

    for p in points_cm:
        plt.plot(*p, "gx")
    
    bboxes = bboxes[labels == 1]
    for box in bboxes:
        x, y, w, h = box
        x1 = int((x - w / 2) * img_w)
        y1 = int((y - h / 2) * img_h)
        x2 = x1 + int(w * img_w)
        y2 = y1 + int(h * img_h)
        
        tmp_bbox_cm = pg_camera.pixel2cm(np.array([
            [x1, y1],
            [x1, y2],
            [x2, y2],
            [x2, y1],
        ]))
        xs = sorted(tmp_bbox_cm[:, 0])
        ys = sorted(tmp_bbox_cm[:, 1])
        
        shape = Rectangle(
            (xs[1], ys[1]),
            xs[2] - xs[1],
            ys[2] - ys[1],
            fill=False, color="b"
        )
        ax.add_patch(shape)
    
    ax.set_xlim(-500, 500)
    ax.set_ylim(-375, 375)
    ax.set_aspect(0.75)

In [91]:
def draw_floorplan_gt(ax, pg_camera, gt_data):
    # plt.xlim(-600, 600)
    # plt.ylim(-300, 300)
    # plt.axis("equal")

    points = gt_data[gt_data["label"] == "chair"][["x", "y"]].values
    points_cm = pg_camera.pixel2cm(points)

    plt.plot(0, 0, "k.", markersize=15)

    for p in points_cm:
        plt.plot(*p, "gx")
    
    bboxes = gt_data[gt_data["label"] == "table"][["xtl", "ytl", "xbr", "ybr"]].values
    for box in bboxes:
        x1, y1, x2, y2 = box
        
        tmp_bbox_cm = pg_camera.pixel2cm(np.array([
            [x1, y1],
            [x1, y2],
            [x2, y2],
            [x2, y1],
        ]))
        xs = sorted(tmp_bbox_cm[:, 0])
        ys = sorted(tmp_bbox_cm[:, 1])
        
        shape = Rectangle(
            (xs[1], ys[1]),
            xs[2] - xs[1],
            ys[2] - ys[1],
            fill=False, color="b"
        )
        ax.add_patch(shape)
    
    ax.set_xlim(-500, 500)
    ax.set_ylim(-375, 375)
    ax.set_aspect(0.75)

In [92]:
def get_list_attributes_by_index(df, idx):
    tmp_row = df.iloc[idx]
    true_columns = tmp_row[tmp_row == True].index.tolist()
    return true_columns

In [97]:
os.makedirs("res", exist_ok=True)

model = YOLO(os.path.join(base_pth, model_name, "weights/best.pt"))

predictions  = {}
for i, row in tqdm(df_imgs.iterrows()):
    gt_objects = df[(df["img_id"] == row["id"]) & (df["label"].isin(classes))]
    gt_objects.reset_index(inplace=True)
    gt_ids = np.arange(len(gt_objects))
    
    img_pth = os.path.join(base_path, "annotation", row["batch"], row["folder"], "images",row["name"])
    pred = model.predict(img_pth, conf=thres, iou=iou_thres, verbose=False)[0]
    img_name, labels, confidences, boxes, centers = process_prediction(pred)
    det_ids = np.arange(len(boxes))
    
    succcess, h, sensor = parse_img_name(img_name)
    if not succcess:
        h = 250
        sensor = "Cam1"
    pg_camera = PgCamera(h * 10 - 700, sensor)
    
    img = cv2.cvtColor(cv2.imread(img_pth), cv2.COLOR_BGR2RGB)
    img_h, img_w, _ = img.shape

    ###########################################################################
    # init evaluator
    evaluator = MetricsEvaluator(
        is_relative_coordinates=True,
        is_xywh_coordinates=True,
        img_size=(INPUT_WIDTH, INPUT_HEIGHT),
        outpath="."
    )

    for i, row in gt_objects.iterrows():
        x_c = (row["xtl"] + row["xbr"]) / 2 / img_w
        y_c = (row["ytl"] + row["ybr"]) / 2 / img_h
        w = (row["xbr"] - row["xtl"]) / img_w
        h = (row["ybr"] - row["ytl"]) / img_h
        evaluator.add_gt_box(
            img_path=img_name,
            class_name=row["label"],
            bbox=[x_c, y_c, w, h],
            box_id=gt_ids[i]
        )

    for box, cl, conf, idx in zip(boxes, labels, confidences, det_ids):
        x_c = box[0] + box[2] / 2
        y_c = box[1] + box[3] / 2
        evaluator.add_det_box(
            img_path=img_name,
            class_name=classes[int(cl)],
            bbox=box,
            confidence=conf,
            box_id=idx
        )

    results = evaluator.eval_conf_vals(0.5)

    metrics = ""
    for result in results:
        precision = result["total TP"] / (result["total TP"] + result["total FP"])
        recall = result["total TP"] / (result["total positives"])

        f_score = 2 * precision * recall / (precision + recall)
        metrics += (
            f"{result['class']}:\n"
            f"    P = {precision:.03f}. R = {recall:.03f}. F1 = {f_score:.03f}\n"
            f"    TP_count = {len(result['TP_ids'])}. FP_count = {len(result['FP_ids'])}. FN_count = {len(result['FN_ids'])}\n"
            f"    FN attributes: {[get_list_attributes_by_index(gt_objects, np.where(gt_ids == i)[0][0]) for i in result['FN_ids']]}\n"
        )
###############################################################################
    
    fig = plt.figure(figsize=(15, 13))
    plt.suptitle(f"{img_name}\n{metrics}", ha="left", x=0.05)
    ax = plt.subplot(2, 2, 1)
    ax.set_title("Input image")
    plt.imshow(img)
    
    ax = plt.subplot(2, 2, 2)
    ax.set_title("Green - gt objects; blue - det `table`, red - det `chair`")
    draw_gt(img, gt_objects)
    draw_pred(img, labels, confidences, boxes)
    plt.imshow(img)
    
    ax = plt.subplot(2, 2, 3)
    ax.set_title("Ground-truth floor plan")
    draw_floorplan_gt(ax, pg_camera, gt_objects);
    
    ax = plt.subplot(2, 2, 4)
    ax.set_title("Predicted floor plan")
    draw_floorplan(ax, pg_camera, img, centers, boxes, labels);

    fig.tight_layout()
    plt.savefig(os.path.join("res", f"{img_name}.png"), dpi=300)
    plt.close()
    # break

6it [00:11,  1.83s/it]/tmp/ipykernel_530902/1856405030.py:64: RuntimeWarning: invalid value encountered in scalar divide
  f_score = 2 * precision * recall / (precision + recall)
28it [00:53,  2.09s/it]/tmp/ipykernel_530902/1856405030.py:62: RuntimeWarning: invalid value encountered in scalar divide
  recall = result["total TP"] / (result["total positives"])
37it [01:10,  1.92s/it]
